# 🤖 LangChain — Solutions Notebook
### Reference solutions for all exercises

> ⚠️ **Instructor use only** — share with students after the lab.


In [8]:
# DO NOT MODIFY
# ── Install dependencies (run once, then restart kernel) ────────────────────
# !pip install --only-binary :all: greenlet
# !pip install langchain langchain-openai langchain-community \
#              langchain-experimental langgraph \
#              duckduckgo-search ddgs

# %matplotlib inline

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_classic.chains import LLMChain, SimpleSequentialChain, SequentialChain, ConversationChain
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_classic.agents import Tool, initialize_agent
from langchain_classic.agents.agent_types import AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_classic.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from typing import TypedDict
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import re

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="gemma-3-4b-it",
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

class DebugSearchTool(DuckDuckGoSearchRun):
    def run(self, query: str) -> str:
        print(f"  [Search] {query[:70]}")
        result = super().run(query)
        print(f"  [Search] {len(result)} chars returned")
        return result

class SafePythonREPL(PythonREPLTool):
    def run(self, query: str) -> str:
        clean = query.strip()
        for m in ["```python","```"]: clean = clean.replace(m,"").strip()
        return super().run(clean)

search_tool = DebugSearchTool()
python_tool = SafePythonREPL()
print("Setup complete")


Setup complete


---
## Solution 1 — First LLM Call + PromptTemplate ⭐

**Key point**: `llm.invoke()` is the modern API replacing the deprecated `.predict()`. It returns a message object; `.content` extracts the text. PromptTemplate separates structure from content — the template is defined once and reused.


In [9]:
# 1a
response = llm.invoke("Explain what a neural network is in exactly one sentence.").content
print("Response:", response)


A neural network is a computer system modeled after the human brain, composed of interconnected nodes (neurons) that process information and learn from data to perform tasks like recognizing patterns or making predictions.Response: A neural network is a computer system modeled after the human brain, composed of interconnected nodes (neurons) that process information and learn from data to perform tasks like recognizing patterns or making predictions.


In [10]:
# 1b
explain_prompt = PromptTemplate.from_template(
    "answering in french. Explain {topic} to {audience} in two sentences. Use a concrete analogy."
)

pairs = [
    {"topic": "gradient descent", "audience": "a 10-year-old"},
    {"topic": "overfitting",      "audience": "a business executive with no ML background"},
]
for p in pairs:
    result = llm.invoke(explain_prompt.format(**p)).content
    print(f"--- {p['topic']} ---")
    print(result)
    print()


Okay, here's an explanation of gradient descent for a 10-year-old, in French and with an analogy:

"Imagine you’re lost on a hilly field and want to get to the lowest point – gradient descent is like taking small steps downhill until you reach it! It works by looking around to see which way is steepest downwards and then taking a little step in that direction." 

---

**Translation of the explanation:**

"Imaginez que vous êtes perdu dans un champ vallonné et voulez atteindre le point le plus bas – la descente par gradient, c'est comme faire de petits pas vers le bas jusqu’à ce que vous l'atteigniez ! Cela fonctionne en regardant autour pour voir quelle direction est la plus raide et puis en faisant un petit pas dans cette direction." 


Would you like me to explain it in a different way or perhaps with another analogy?--- gradient descent ---
Okay, here's an explanation of gradient descent for a 10-year-old, in French and with an analogy:

"Imagine you’re lost on a hilly field and wan

APIError: Model reloaded.

---
## Solution 2 — Three-Step SequentialChain ⭐⭐

**Key point**: `output_key` names each chain's output. `SequentialChain` passes all named outputs forward — later chains can reference earlier results by name. Always set `verbose=True` during development to see intermediate outputs.


In [ ]:
chain_fact = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(
        "Write 4 interesting sentences about the science of {topic}."
    ),
    output_key="fact",
)
chain_translate = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(
        "Translate the following text to Italian:\n{fact}"
    ),
    output_key="fact_it",
)
chain_tweet = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(
        "Write a Twitter post (max 280 chars, 2 relevant hashtags) from this Italian text:\n{fact_it}"
    ),
    output_key="tweet",
)
science_pipeline = SequentialChain(
    chains=[chain_fact, chain_translate, chain_tweet],
    input_variables=["topic"],
    output_variables=["fact", "fact_it", "tweet"],
    verbose=True,
)
result = science_pipeline({"topic": "black holes"})
print("\nFact (EN):\n", result["fact"])
print("\nFact (IT):\n", result["fact_it"])
print("\nTweet:\n", result["tweet"])


---
## Solution 3 — Custom Tools ⭐

**Key point**: the `@tool` docstring IS the tool's description — the LLM reads it to decide when and how to call the tool. Specific input format instructions ('a numeric string, e.g. 42.195') are not just documentation — they are instructions to the model. Always test tools with `.run()` before giving them to an agent.


In [8]:
@tool
def km_to_miles(km: str) -> str:
    """Converts a distance from kilometres to miles.
    Input: a numeric string representing kilometres, e.g. '42.195'.
    Output: a string stating the equivalent distance in miles."""
    try:
        return f"{float(km)} km = {float(km) * 0.621371:.4f} miles"
    except ValueError:
        return "Error: input must be a numeric string, e.g. '10' or '42.195'"

@tool
def fibonacci(n: str) -> str:
    """Returns the first n Fibonacci numbers as a comma-separated string.
    Input: a positive integer string, e.g. '8'.
    Output: the Fibonacci sequence up to n terms, e.g. '0, 1, 1, 2, 3, 5, 8, 13'."""
    try:
        n_int = int(n)
        if n_int <= 0: return "Error: n must be positive."
        seq = [0, 1]
        while len(seq) < n_int: seq.append(seq[-1] + seq[-2])
        return ", ".join(str(x) for x in seq[:n_int])
    except ValueError:
        return "Error: input must be a positive integer string."

print(km_to_miles.run("42.195"))
print(fibonacci.run("8"))


42.195 km = 26.2187 miles
0, 1, 1, 2, 3, 5, 8, 13


---
## Solution 4 — Multi-Tool Agent ⭐⭐

**Key point**: wrap built-in tools in `Tool()` to write custom descriptions. `@tool`-decorated functions can be passed directly. Read the verbose trace: find every `Thought`, `Action` (JSON), and `Observation`. This is the ReAct loop made visible.


In [9]:
all_tools = [
    Tool(name="Web Search", func=search_tool.run,
         description="Find current facts, statistics, or news from the internet."),
    Tool(name="Python Executor", func=python_tool.run,
         description="Execute Python code for calculations, plots, or text processing."),
    km_to_miles,
    fibonacci,
]
agent_ex4 = initialize_agent(
    tools=all_tools, llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, handle_parsing_errors=True, max_iterations=8,
)
result = agent_ex4.run(
    "The distance from Rome to Milan is 572 km. "
    "Convert it to miles using the km_to_miles tool, "
    "compute the first 8 Fibonacci numbers using the fibonacci tool, "
    "then print both results side by side using Python."
)
print("\nFinal Answer:", result)


C:\Users\kiki_\AppData\Local\Temp\ipykernel_5092\2641607020.py:9: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_ex4 = initialize_agent(
C:\Users\kiki_\AppData\Local\Temp\ipykernel_5092\2641607020.py:14: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent_ex4.run(




> Entering new AgentExecutor chain...
Action: ```json
{
  "action": "km_to_miles",
  "action_input": "572"
}
```
Action: ```json
{
  "action": "km_to_miles",
  "action_input": "572"
}
```

Observation: 572.0 km = 355.4242 miles
Thought:Action: ```json
{
  "action": "fibonacci",
  "action_input": "8"
}
```Action: ```json
{
  "action": "fibonacci",
  "action_input": "8"
}
```
Observation: 0, 1, 1, 2, 3, 5, 8, 13
Thought:Action: ```json
{
  "action": "Python Executor",
  "action_input": "print(\"Miles from Rome to Milan:\", 355.4242) \nprint(\"First 8 Fibonacci numbers:\", \"0, 1, 1, 2, 3, 5, 8, 13\")"
}
```

Python REPL can execute arbitrary code. Use with caution.


Action: ```json
{
  "action": "Python Executor",
  "action_input": "print(\"Miles from Rome to Milan:\", 355.4242) \nprint(\"First 8 Fibonacci numbers:\", \"0, 1, 1, 2, 3, 5, 8, 13\")"
}
```
Observation: Miles from Rome to Milan: 355.4242
First 8 Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13

Thought:Action: ```json
{
  "action": "Final Answer",
  "action_input": "Miles from Rome to Milan: 355.4242\nFirst 8 Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13"
}
```Action: ```json
{
  "action": "Final Answer",
  "action_input": "Miles from Rome to Milan: 355.4242\nFirst 8 Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13"
}
```

> Finished chain.

Final Answer: Miles from Rome to Milan: 355.4242
First 8 Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13


---
## Solution 5 — Memory Comparison ⭐⭐

**Key point**: `ConversationBufferMemory` retains all turns — after 5 turns it knows name, city, language, and thesis topic. `WindowMemory(k=2)` only retains the last 2 turns — after 5 turns it has forgotten turns 1 and 2, so it doesn't know the name or city. This is the core token-cost vs. context-retention trade-off.


In [10]:
turns = [
    "My name is Giulia.",
    "I study computer science in Rome.",
    "My favourite language is Python.",
    "I am writing my thesis on federated learning.",
    "Summarise everything you know about me.",
]

# Buffer agent
buf_mem = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
agent_buf = initialize_agent(
    tools=[Tool(name="Web Search", func=search_tool.run,
                description="Find facts from the web.")],
    llm=llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    memory=buf_mem, verbose=False, handle_parsing_errors=True, max_iterations=3
)
buf_responses = [agent_buf.run(t) for t in turns]

# Window agent k=2
win_mem = ConversationBufferWindowMemory(memory_key="chat_history", return_messages=True, k=2)
agent_win = initialize_agent(
    tools=[Tool(name="Web Search", func=search_tool.run,
                description="Find facts from the web.")],
    llm=llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    memory=win_mem, verbose=False, handle_parsing_errors=True, max_iterations=3
)
win_responses = [agent_win.run(t) for t in turns]

print("=" * 60)
print("Turn 5: 'Summarise everything you know about me.'")
print("=" * 60)
print(f"BUFFER: {buf_responses[-1]}")
print(f"WINDOW: {win_responses[-1]}")
print()
print("Explanation:")
print("Buffer remembers all 4 prior turns -> knows name, city, language, thesis.")
print("Window(k=2) only has turns 3+4 -> forgot name (turn1) and city (turn2).")


C:\Users\kiki_\AppData\Local\Temp\ipykernel_5092\3091465255.py:10: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  buf_mem = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


Action: ```{ "action": "Final Answer", "action_input": "Hello! My name is Giulia." }```Action: ```{ "action": "Final Answer", "action_input": "That's fantastic! Studying computer science in Rome sounds like a unique and exciting experience." }```Action: ```{ "action": "Final Answer", "action_input": "Python is an excellent choice! It's a versatile and popular programming language known for its readability and extensive libraries." }```Action: ```{ "action": "Web Search", "action_input": "federated learning thesis" }```  [Search] federated learning thesis


DDGSException: RuntimeError: RuntimeError('error sending request for url (https://leta.mullvad.net/search/__data.json?q=federated+learning+thesis&engine=brave&x-sveltekit-invalidated=001&country=wt&lang=wt&lastUpdated=y): client error (Connect)\n\nCaused by:\n    0: client error (Connect)\n    1: dns error: Host sconosciuto. (os error 11001)\n    2: Host sconosciuto. (os error 11001)')

---
## Solution 6 — Two-Agent Pipeline ⭐⭐⭐

**Key point**: the Summariser has NO tools because it doesn't need them. Its only job is to reason over text it receives in the prompt. Reasoning-only agents are cheaper (fewer tokens) and faster. Prompt specificity matters: 'EXACTLY one sentence + 3 bullets + 1 question' produces consistent structured output; vague prompts produce variable format.


In [ ]:
res_mem = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
researcher = initialize_agent(
    tools=[Tool(name="Web Search", func=search_tool.run,
                description="Find information, facts, and recent developments on any topic.")],
    llm=llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    memory=res_mem, verbose=True, handle_parsing_errors=True, max_iterations=4
)

sum_mem = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
summariser = initialize_agent(
    tools=[],
    llm=llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    memory=sum_mem, verbose=True, handle_parsing_errors=True, max_iterations=2
)

def run_research_pipeline(topic: str) -> None:
    print(f"\nTopic: {topic}\n")
    research_output = researcher.run(
        f"Search for comprehensive information about '{topic}', "
        f"including recent developments and key concepts."
    )
    display(Markdown(f"**Research:**\n{research_output}"))
    summary = summariser.run(
        f"Based on this research about '{topic}':\n{research_output}\n\n"
        "Produce a structured summary with EXACTLY:\n"
        "1. One-sentence definition\n"
        "2. Three key facts as bullet points (-)\n"
        "3. One open research question"
    )
    display(Markdown(f"**Summary:**\n{summary}"))

run_research_pipeline("Transformer neural networks")


---
## Solution 7 — Fault-Tolerant Orchestrator ⭐⭐⭐

**Key point**: production agents ALWAYS fail sometimes. The three mechanisms here cover the three most common failure modes: insufficient output (retry), malformed output (fallback), and unexpected exceptions (try/except). The `has_bullet_list` heuristic is intentionally simple — perfect validation is impossible; good-enough validation is always achievable.


In [ ]:
def has_bullet_list(text: str) -> bool:
    return bool(re.search(r'(^\s*[-*]|^\s*\d+\.)', text, re.MULTILINE))

def run_robust_pipeline(topic: str) -> None:
    print(f"\nTopic: {topic}\n")

    # Stage 1: research with retry
    try:
        research_output = researcher.run(
            f"Search for information about: '{topic}'."
        )
        if len(research_output.strip()) < 80:
            print(f"  Research too short ({len(research_output)} chars) -- retrying...")
            research_output = researcher.run(
                f"Give a detailed explanation of: {topic}"
            )
    except Exception as e:
        print(f"  Researcher failed: {e}")
        return

    display(Markdown(f"**Research ({len(research_output)} chars):**\n{research_output}"))

    # Stage 2: summarise with validation + fallback
    try:
        summary = summariser.run(
            f"Summarise the following with bullet points:\n{research_output}"
        )
    except Exception as e:
        print(f"  Summariser failed: {e} -- using LLM fallback")
        summary = llm.invoke(f"List 3 key facts about: {topic}").content
    else:
        if not has_bullet_list(summary):
            print("  No bullet list -- using LLM fallback")
            summary = llm.invoke(f"List 3 key facts about: {topic}").content

    display(Markdown(f"**Summary:**\n{summary}"))

run_robust_pipeline("mRNA vaccines")


---
## Solution 8 — LangGraph Two-Node Graph ⭐⭐

**Key point**: nodes are pure functions — they receive State and return a dict of updates. They do NOT return the entire State. LangGraph merges the update automatically. `compile()` validates the graph at construction time: unreachable nodes or missing entry points raise errors immediately, not at runtime.


In [ ]:
class FactState(TypedDict):
    topic:   str
    english: str
    italian: str

def node_generate(state: FactState) -> dict:
    print("  [Node: generate]")
    prompt = f"Write 3 interesting science facts about {state['topic']}."
    return {"english": llm.invoke(prompt).content}

def node_translate(state: FactState) -> dict:
    print("  [Node: translate]")
    prompt = f"Translate this to Italian:\n{state['english']}"
    return {"italian": llm.invoke(prompt).content}

g8 = StateGraph(FactState)
g8.add_node("generate",  node_generate)
g8.add_node("translate", node_translate)
g8.set_entry_point("generate")
g8.add_edge("generate",  "translate")
g8.add_edge("translate", END)
graph8 = g8.compile()

result8 = graph8.invoke({"topic": "quantum computing"})
print("English:\n", result8["english"])
print("\nItalian:\n", result8["italian"])


---
## Solution 9 — LangGraph Conditional Edge + Retry ⭐⭐⭐

**Key point**: `add_conditional_edges` takes a routing function that returns a string key. The dictionary maps those keys to destination nodes. Setting threshold=500 forces the retry to fire on the first attempt since a 3-fact output is typically 200-400 chars. The `attempts` counter prevents infinite loops — always include a loop-breaking condition.


In [ ]:
class FactStateV2(TypedDict):
    topic:    str
    english:  str
    italian:  str
    attempts: int

def node_generate_v2(state: FactStateV2) -> dict:
    attempts = state.get("attempts", 0)
    print(f"  [Node: generate] attempt {attempts + 1}")
    detail = "very detailed" if attempts > 0 else "interesting"
    prompt = f"Write 3 {detail} science facts about {state['topic']}."
    return {"english": llm.invoke(prompt).content, "attempts": attempts + 1}

def router(state: FactStateV2) -> str:
    if len(state["english"]) < 500 and state.get("attempts", 0) < 2:
        print(f"  [Router] Too short ({len(state['english'])} chars) -> retry")
        return "retry"
    print(f"  [Router] OK ({len(state['english'])} chars) -> translate")
    return "ok"

g9 = StateGraph(FactStateV2)
g9.add_node("generate_v2", node_generate_v2)
g9.add_node("translate",   node_translate)   # reuse from Ex 8
g9.set_entry_point("generate_v2")
g9.add_conditional_edges(
    "generate_v2", router,
    {"retry": "generate_v2", "ok": "translate"}
)
g9.add_edge("translate", END)
graph9 = g9.compile()

result9 = graph9.invoke({"topic": "quantum computing", "attempts": 0})
print(f"\nAttempts: {result9['attempts']}")
print(f"English ({len(result9['english'])} chars):\n{result9['english']}")
print(f"\nItalian:\n{result9['italian']}")


---
## 💡 Instructor Notes

| Exercise | Common Mistakes | Discussion Points |
|----------|-----------------|-------------------|
| 1 | Using `.predict()` instead of `.invoke().content` | Deprecation as a design signal: API evolution matters |
| 2 | Wrong `output_key` / `input_variables` names | Named pipes pattern; exact match required |
| 3 | Vague docstring → agent ignores or misuses tool | Docstring = prompt; treat it with same care as prompts |
| 4 | Not reading the verbose trace carefully | Reading the ReAct loop builds intuition for debugging |
| 5 | Not explaining WHY window forgets | Token cost is a first-class engineering concern |
| 6 | Summariser prompt too vague → inconsistent structure | Specificity in output format instructions is critical |
| 7 | Missing try/except OR missing fallback | Two separate failure modes; both need handling |
| 8 | Returning full state instead of update dict | Nodes return diffs, not full state — key LangGraph concept |
| 9 | No loop-breaking condition → infinite retry | Always include a max_attempts check in routers |

**Grading suggestion**: Exercises 1–4 are self-testing (run and see output). Exercises 5 and 8 require conceptual explanation. Exercises 6, 7, and 9 require manual evaluation of prompt quality and fault-tolerance logic.
